In [ ]:
# from mdgen.parsing import parse_train_args
# args = parse_train_args()

import glob
sim_ckpt = glob.glob("workdir/default/epoch=10959-step=0668560-val_loss=1.2114.ckpt")[0]
device = "cuda"

In [ ]:
import os, torch, tqdm, time
import numpy as np
from mdgen.equivariant_wrapper import EquivariantMDGenWrapper

In [ ]:
# out_dir = "experiments/ala-dipeptide/data_consistency/sde_step100/s1-e2880/"
out_dir = "experiments/MP_C/e10959/"
# out_dir = "experiments/ala-dipeptide/data_consistency/vardecreasing_sde_step100/s1-e259/"

os.makedirs(out_dir, exist_ok=True)
with open(f"{out_dir}/README.md", "w") as fp:
    fp.write(sim_ckpt)

In [ ]:
ckpt = torch.load(sim_ckpt, weights_only=False)
hparams = ckpt["hyper_parameters"]
args = hparams['args']
# args.sampling_method = "euler"
args.inference_steps = 100
args.data_dir = "data/MP_C_data/"
# args.likelihood = "FND"

In [ ]:

from mdgen.dataset import EquivariantTransformerDataset_MaterialProject
dataset = EquivariantTransformerDataset_MaterialProject(args.data_dir, args.cutoff, species=[6], sim_condition=False, stage="train")


In [ ]:

model = EquivariantMDGenWrapper(**hparams)
print(model.model)
model.load_state_dict(ckpt["state_dict"], strict=False)
model.eval().to(device)

In [ ]:
print(model.args)
print(model.args.path_type)
print(model.args.sampling_method)
print(model.args.inference_steps)
print(model.args.likelihood)
print(model.args.x0std)

In [ ]:
batch_size = 1
val_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size,
    num_workers=0,
    shuffle=True,
)
sample_batch = next(iter(val_loader))


In [ ]:
print(sample_batch.keys())

In [ ]:
print(dataset[0]["x"].shape)

## Test generative model

In [ ]:
for key in ['species', 'x', 'cell', 'num_atoms', 'mask', 'v_mask']:
    try:
        sample_batch[key] = sample_batch[key].to(device)
    except:
        print(f"{key} not found")


# vector_out, aa_out, zs, logprob_samples, _logprob_samples = model.inference(sample_batch)
# print(logprob_samples.shape, _logprob_samples.shape, vector_out.shape, zs.shape)
vector_out, aa_out, cell_out = model.inference(sample_batch)

In [ ]:
# print(logprob_samples)
# print(_logprob_samples)
# print(_logprob_samples - logprob_samples)
# print(torch.concatenate([logprob_samples, _logprob_samples], dim=-1))

In [ ]:
@torch.no_grad()
def rollout(model, batch):
    logp, positions, _ = model.inference(batch)
    new_batch = {**batch}
    new_batch['x'] = positions
    return logp, positions, new_batch


map_to_chemical_symbol = {
    0: "H",
    1: 'C',
    2: "N",
    3: "O"

}

In [ ]:
print(len(dataset))

In [ ]:

idx_rollouts = np.arange(len(dataset))
# np.random.shuffle(idx_rollouts)
# idx_rollouts = idx_rollouts[:500]

In [ ]:
print(idx_rollouts)

In [ ]:
from ase import Atoms
from ase.geometry.geometry import get_distances
import shutil, os
from ase.io import write

all_rollout_atoms_ref_0 = []
all_rollout_atoms = []
all_rollout_atoms_ref = []
start = time.time()
all_logp = []
for i_rollout in range(0, len(idx_rollouts)):
    idx = idx_rollouts[i_rollout]
    print(i_rollout, idx)
    filename = os.path.join(out_dir, f"gentraj_{idx}.xyz")
    filename_ref = os.path.join(out_dir, f"reftraj_{idx}.xyz")
    if os.path.exists(filename):
        os.remove(filename)
        os.remove(filename_ref)
    # fout_cv = open(os.path.join(out_dir, f"CV_{idx}.txt"), "a")
    # fout_logp = open(os.path.join(out_dir, f"Logp_{idx}.txt"), "a")
    # fout_reverse_logp = open(os.path.join(out_dir, f"reverse_Logp_{idx}.txt"), "a")
    # fout_zs = open(os.path.join(out_dir, f"Uzs_{idx}.txt"), "a")
    for i_sample in range(1):
        item = dataset.__getitem__(idx)
        batch = next(iter(torch.utils.data.DataLoader([item])))

        for key in ['species', 'x', 'cell', 'num_atoms', 'mask', 'v_mask']:
            try:
                batch[key] = batch[key].to(device)
            except:
                print(f"{key} not found")
        # np.savetxt(fout_cv, batch['cv'].squeeze(0).cpu().numpy())
        # fout_cv.flush()

        # logp, pred_pos, _, reverse_logp, zs, pred_zs = model.inference(batch)
        pred_frac_pos, _, cell_out = model.inference(batch)
        pred_pos = pred_frac_pos[0][0] @ cell_out[0][0]
        # np.savetxt(fout_logp, logp.detach().cpu().numpy() )
        # fout_logp.flush()
        # np.savetxt(fout_reverse_logp, reverse_logp.detach().cpu().numpy() )
        # fout_reverse_logp.flush()
        # np.savetxt(fout_zs, [[(zs**2/2).sum().detach().cpu().numpy(), (pred_zs**2/2).sum().detach().cpu().numpy()]])
        # fout_zs.flush()

        labels = torch.argmax(batch["species"], dim=3).squeeze(0)
        symbols = [[map_to_chemical_symbol[int(i_elem.to('cpu'))] for i_elem in labels[i_conf]] for i_conf in range(len(labels))]
        all_atoms = []
        all_atoms_ref = []
        t = 0
        print("rollout", i_rollout, "idx = ", idx+i_sample, "t", t)
        formula = "".join(symbols[t])
        atoms = Atoms(formula, positions=pred_pos.detach().cpu().numpy(), cell=cell_out[0][0].detach().cpu().numpy(), pbc=[1,1,1])
        # atoms.set_chemical_symbols(symbols[t])
        all_atoms.append(atoms)
        atoms_ref = Atoms(formula, positions=batch["x"][0][t].cpu().numpy(), cell=batch['cell'][0][0].cpu().numpy(), pbc=[1,1,1])
        all_atoms_ref.append(atoms_ref)

        for atoms in all_atoms:
            write(filename, atoms, append=True)
        for ref_atoms in all_atoms_ref:
            write(filename_ref, ref_atoms, append=True)
        
        del pred_frac_pos
        del pred_pos
        del cell_out